In [ ]:
# !pip install yfinance pandas numpy matplotlib scipy ta-lib vectorbt

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import talib
import vectorbt as vbt
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os, sys

In [ ]:
# ============================================================
# SUPERTREND STRATEGY — Configuration
# ============================================================
TICKER = 'QQQ'
START_DATE = '2018-01-01'
TRAIN_RATIO = 0.60

# Backtest settings
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005

print(f"Ticker: {TICKER}")
print(f"Start: {START_DATE} | Train ratio: {TRAIN_RATIO}")
print(f"Capital: ${INIT_CASH:,} | Fees: {FEES:.2%} | Slippage: {SLIPPAGE:.2%}")

In [ ]:
# ============================================================
# Download Data
# ============================================================
df = yf.download(TICKER, start=START_DATE, auto_adjust=True)
close = df['Close'].squeeze()
high = df['High'].squeeze()
low = df['Low'].squeeze()

split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx]
val_close = close.iloc[split_idx:]

print(f"Total bars: {len(close)} ({close.index[0].strftime('%Y-%m-%d')} \u2192 {close.index[-1].strftime('%Y-%m-%d')})")
print(f"Train: {len(train_close)} bars ({train_close.index[0].strftime('%Y-%m-%d')} \u2192 {train_close.index[-1].strftime('%Y-%m-%d')})")
print(f"Test:  {len(val_close)} bars ({val_close.index[0].strftime('%Y-%m-%d')} \u2192 {val_close.index[-1].strftime('%Y-%m-%d')})")
print(f"\n{df.tail(5)}")

## Supertrend Strategy

**Indicator**: Supertrend is an ATR-based trend-following overlay:
- `Upper Band = (High + Low)/2 + Multiplier × ATR(period)`
- `Lower Band = (High + Low)/2 - Multiplier × ATR(period)`
- The Supertrend line flips between bands based on price crossovers

**Entry (Long)**: Close crosses above the Supertrend line (trend turns bullish)
**Exit**: Close crosses below the Supertrend line (trend turns bearish)

**Parameters to optimize**:
- `atr_period`: ATR lookback (7–30)
- `multiplier`: ATR multiplier for band width (1.0–4.0)

In [ ]:
# ============================================================
# Supertrend Indicator
# ============================================================
def supertrend(high, low, close, atr_period=10, multiplier=3.0):
    """
    Compute Supertrend indicator.
    Returns: supertrend_line (Series), direction (Series: 1=bullish, -1=bearish)
    """
    atr = talib.ATR(high, low, close, timeperiod=atr_period)
    hl2 = (high + low) / 2
    
    upper_band = hl2 + multiplier * atr
    lower_band = hl2 - multiplier * atr
    
    st = pd.Series(np.nan, index=close.index)
    direction = pd.Series(1, index=close.index)
    
    for i in range(atr_period, len(close)):
        # Carry forward bands (tighten only)
        if i > atr_period:
            if lower_band.iloc[i] > lower_band.iloc[i-1] or close.iloc[i-1] < lower_band.iloc[i-1]:
                pass  # use current lower_band
            else:
                lower_band.iloc[i] = lower_band.iloc[i-1]
            
            if upper_band.iloc[i] < upper_band.iloc[i-1] or close.iloc[i-1] > upper_band.iloc[i-1]:
                pass  # use current upper_band
            else:
                upper_band.iloc[i] = upper_band.iloc[i-1]
        
        # Determine direction
        if i > atr_period:
            if st.iloc[i-1] == upper_band.iloc[i-1]:
                direction.iloc[i] = -1 if close.iloc[i] <= upper_band.iloc[i] else 1
            else:
                direction.iloc[i] = 1 if close.iloc[i] >= lower_band.iloc[i] else -1
        
        st.iloc[i] = lower_band.iloc[i] if direction.iloc[i] == 1 else upper_band.iloc[i]
    
    return st, direction

# Test with default params
st_line, st_dir = supertrend(high, low, close, atr_period=10, multiplier=3.0)

# Show recent values
test_df = pd.DataFrame({
    'Close': close,
    'Supertrend': st_line,
    'Direction': st_dir
}).tail(10)
print("Supertrend (default ATR=10, Mult=3.0):")
print(test_df.to_string())

In [ ]:
# ============================================================
# Parameter Ranges
# ============================================================
atr_range = list(range(7, 31, 1))        # 7 to 30
mult_range = [round(x * 0.1, 1) for x in range(10, 41, 2)]  # 1.0 to 4.0 step 0.2

total_combos = len(atr_range) * len(mult_range)
print(f"ATR periods: {atr_range[0]}\u2013{atr_range[-1]} ({len(atr_range)} values)")
print(f"Multipliers: {mult_range[0]}\u2013{mult_range[-1]} ({len(mult_range)} values)")
print(f"Total combinations: {total_combos}")

In [ ]:
# ============================================================
# Grid Search (In-Sample)
# ============================================================
results = []
train_high = high.iloc[:split_idx]
train_low = low.iloc[:split_idx]

for i, atr_p in enumerate(atr_range):
    for mult in mult_range:
        try:
            st_line, st_dir = supertrend(train_high, train_low, train_close, atr_p, mult)
            
            # Entry: direction flips to bullish (1-bar delay)
            entries_raw = (st_dir == 1) & (st_dir.shift(1) == -1)
            exits_raw = (st_dir == -1) & (st_dir.shift(1) == 1)
            entries = entries_raw.shift(1).fillna(False)
            exits = exits_raw.shift(1).fillna(False)
            
            pf = vbt.Portfolio.from_signals(
                train_close, entries, exits,
                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
            )
            
            n_trades = pf.trades.count()
            if n_trades < 5:
                continue
            
            results.append({
                'atr_period': atr_p,
                'multiplier': mult,
                'is_sharpe': pf.sharpe_ratio(),
                'is_return': pf.total_return(),
                'is_max_dd': pf.max_drawdown(),
                'is_trades': n_trades,
                'is_win_rate': pf.trades.win_rate(),
                'is_profit_factor': pf.trades.profit_factor(),
            })
        except Exception:
            continue
    
    if (i + 1) % 5 == 0:
        print(f"\ud83d\udd04 [{(i+1)*len(mult_range)}/{total_combos} combos]")

results_df = pd.DataFrame(results).sort_values('is_sharpe', ascending=False)
print(f"\n\u2705 Completed: {len(results_df)} valid configs (min 5 trades)")
print(f"\nTop 10 by Sharpe:")
print(results_df.head(10).to_string(index=False))

In [ ]:
# ============================================================
# Out-of-Sample Validation (Top 5)
# ============================================================
val_high = high.iloc[split_idx:]
val_low = low.iloc[split_idx:]
top5 = results_df.head(5)

oos_results = []
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for idx, (_, row) in enumerate(top5.iterrows()):
    atr_p = int(row['atr_period'])
    mult = row['multiplier']
    
    st_line, st_dir = supertrend(val_high, val_low, val_close, atr_p, mult)
    entries_raw = (st_dir == 1) & (st_dir.shift(1) == -1)
    exits_raw = (st_dir == -1) & (st_dir.shift(1) == 1)
    entries = entries_raw.shift(1).fillna(False)
    exits = exits_raw.shift(1).fillna(False)
    
    pf = vbt.Portfolio.from_signals(
        val_close, entries, exits,
        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
    )
    
    oos_sharpe = pf.sharpe_ratio()
    oos_return = pf.total_return()
    degradation = 1 - (oos_sharpe / row['is_sharpe']) if row['is_sharpe'] > 0 else np.nan
    
    oos_results.append({
        'atr_period': atr_p,
        'multiplier': mult,
        'is_sharpe': row['is_sharpe'],
        'oos_sharpe': oos_sharpe,
        'degradation': degradation,
        'oos_return': oos_return,
        'oos_max_dd': pf.max_drawdown(),
        'oos_trades': pf.trades.count(),
    })
    
    # Plot equity curve
    eq = pf.value()
    axes[idx].plot(eq.index, eq.values, color='#4A90D9')
    axes[idx].set_title(f"ATR={atr_p} M={mult}\nOOS Sharpe={oos_sharpe:.2f}", fontsize=9)
    axes[idx].tick_params(labelsize=7)
    if idx == 0:
        axes[idx].set_ylabel('Portfolio Value ($)')

plt.suptitle(f'{TICKER} \u2014 Supertrend OOS Validation (Top 5)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

oos_df = pd.DataFrame(oos_results)
print("\nIS vs OOS Comparison:")
print(oos_df.to_string(index=False))

# Select best OOS
best = oos_df.sort_values('oos_sharpe', ascending=False).iloc[0]
BEST_ATR = int(best['atr_period'])
BEST_MULT = best['multiplier']
print(f"\n\u2705 Best OOS config: ATR={BEST_ATR}, Multiplier={BEST_MULT}")
print(f"   IS Sharpe={best['is_sharpe']:.2f} \u2192 OOS Sharpe={best['oos_sharpe']:.2f} (degradation: {best['degradation']:.1%})")

In [ ]:
# ============================================================
# Parameter Sensitivity Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ATR sensitivity (fix multiplier at best)
atr_sens = results_df[results_df['multiplier'] == BEST_MULT].sort_values('atr_period')
if len(atr_sens) > 0:
    colors = ['#2ECC71' if s > 0 else '#E74C3C' for s in atr_sens['is_sharpe']]
    axes[0].bar(atr_sens['atr_period'], atr_sens['is_sharpe'], color=colors, alpha=0.8)
    axes[0].axvline(BEST_ATR, color='#4A90D9', linestyle='--', linewidth=2, label=f'Best={BEST_ATR}')
    axes[0].set_xlabel('ATR Period')
    axes[0].set_ylabel('IS Sharpe Ratio')
    axes[0].set_title(f'ATR Sensitivity (Mult={BEST_MULT})')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Multiplier sensitivity (fix ATR at best)
mult_sens = results_df[results_df['atr_period'] == BEST_ATR].sort_values('multiplier')
if len(mult_sens) > 0:
    colors = ['#2ECC71' if s > 0 else '#E74C3C' for s in mult_sens['is_sharpe']]
    axes[1].bar(mult_sens['multiplier'], mult_sens['is_sharpe'], color=colors, alpha=0.8, width=0.15)
    axes[1].axvline(BEST_MULT, color='#4A90D9', linestyle='--', linewidth=2, label=f'Best={BEST_MULT}')
    axes[1].set_xlabel('Multiplier')
    axes[1].set_ylabel('IS Sharpe Ratio')
    axes[1].set_title(f'Multiplier Sensitivity (ATR={BEST_ATR})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{TICKER} \u2014 Supertrend Parameter Sensitivity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Full-Sample Backtest (Best Parameters)
# ============================================================
st_line, st_dir = supertrend(high, low, close, BEST_ATR, BEST_MULT)

entries_raw = (st_dir == 1) & (st_dir.shift(1) == -1)
exits_raw = (st_dir == -1) & (st_dir.shift(1) == 1)
entries = entries_raw.shift(1).fillna(False)
exits = exits_raw.shift(1).fillna(False)

pf = vbt.Portfolio.from_signals(
    close, entries, exits,
    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
)

# Metrics
total_ret = pf.total_return()
n_years = len(close) / 252
cagr = (1 + total_ret) ** (1/n_years) - 1
sharpe = pf.sharpe_ratio()
sortino = pf.sortino_ratio()
max_dd = pf.max_drawdown()
calmar = cagr / abs(max_dd) if max_dd != 0 else 0
n_trades = pf.trades.count()
win_rate = pf.trades.win_rate()
pf_ratio = pf.trades.profit_factor()

# Buy & hold comparison
bh = vbt.Portfolio.from_holding(close, init_cash=INIT_CASH, freq='D')
bh_ret = bh.total_return()
bh_sharpe = bh.sharpe_ratio()

print(f"{'='*55}")
print(f"SUPERTREND STRATEGY \u2014 Full Sample Results")
print(f"{'='*55}")
print(f"Params: ATR={BEST_ATR}, Multiplier={BEST_MULT}")
print(f"Period: {close.index[0].strftime('%Y-%m-%d')} \u2192 {close.index[-1].strftime('%Y-%m-%d')}")
print(f"{'\u2500'*55}")
print(f"Total Return:   {total_ret:>10.1%}  (B&H: {bh_ret:.1%})")
print(f"CAGR:           {cagr:>10.1%}")
print(f"Sharpe:         {sharpe:>10.2f}  (B&H: {bh_sharpe:.2f})")
print(f"Sortino:        {sortino:>10.2f}")
print(f"Max Drawdown:   {max_dd:>10.1%}")
print(f"Calmar:         {calmar:>10.2f}")
print(f"Trades:         {n_trades:>10d}")
print(f"Win Rate:       {win_rate:>10.1%}")
print(f"Profit Factor:  {pf_ratio:>10.2f}")

In [ ]:
# ============================================================
# Performance Dashboard
# ============================================================
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 1, height_ratios=[3, 1.2, 1], hspace=0.08)

STRAT_COLOR = '#4A90D9'
BENCH_COLOR = '#999999'
DD_COLOR = '#FF6B6B'
BUY_COLOR = '#2ECC71'
SELL_COLOR = '#E74C3C'

# --- Panel 1: Equity + Supertrend overlay ---
ax1 = fig.add_subplot(gs[0])
eq = pf.value()
bh_eq = bh.value()
ax1.plot(eq.index, eq.values, color=STRAT_COLOR, linewidth=1.5,
         label=f'Supertrend  Sharpe={sharpe:.2f}  Ret={total_ret:.1%}')
ax1.plot(bh_eq.index, bh_eq.values, color=BENCH_COLOR, linewidth=1, linestyle='--',
         label=f'Buy & Hold  Sharpe={bh_sharpe:.2f}  Ret={bh_ret:.1%}')
ax1.set_ylabel('Portfolio Value ($)', fontsize=11)
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.tick_params(labelbottom=False)
ax1.set_title(
    f'{TICKER} \u2014 Supertrend Strategy | ATR={BEST_ATR} Mult={BEST_MULT}\n'
    f'CAGR: {cagr:.1%}  Sharpe: {sharpe:.2f}  MaxDD: {max_dd:.1%}  Trades: {n_trades}',
    fontsize=13, fontweight='bold'
)

# --- Panel 2: Price + Supertrend line with buy/sell markers ---
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.plot(close.index, close.values, color='black', linewidth=0.8, label='Close')

# Color Supertrend line by direction
bull_mask = st_dir == 1
bear_mask = st_dir == -1
ax2.plot(close.index[bull_mask], st_line[bull_mask], color=BUY_COLOR, linewidth=1, label='ST Bullish')
ax2.plot(close.index[bear_mask], st_line[bear_mask], color=SELL_COLOR, linewidth=1, label='ST Bearish')

# Buy/sell markers
buy_dates = entries[entries].index
sell_dates = exits[exits].index
ax2.scatter(buy_dates, close[buy_dates], marker='^', color=BUY_COLOR, s=30, zorder=5, label='Buy')
ax2.scatter(sell_dates, close[sell_dates], marker='v', color=SELL_COLOR, s=30, zorder=5, label='Sell')

ax2.set_ylabel('Price ($)', fontsize=10)
ax2.legend(loc='upper left', fontsize=7, ncol=3)
ax2.grid(True, alpha=0.3)
ax2.tick_params(labelbottom=False)

# --- Panel 3: Drawdown ---
ax3 = fig.add_subplot(gs[2], sharex=ax1)
dd = pf.drawdown() * 100
ax3.fill_between(dd.index, dd.values, 0, color=DD_COLOR, alpha=0.4)
ax3.set_ylabel('Drawdown (%)', fontsize=10)
ax3.set_xlabel('Date', fontsize=11)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Trade Analysis
# ============================================================
trades = pf.trades.records_readable

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Trade returns distribution
trade_rets = trades['Return'].values * 100
winners = trade_rets[trade_rets > 0]
losers = trade_rets[trade_rets <= 0]
axes[0,0].hist(winners, bins=20, color='#2ECC71', alpha=0.7, label=f'Winners ({len(winners)})')
axes[0,0].hist(losers, bins=20, color='#E74C3C', alpha=0.7, label=f'Losers ({len(losers)})')
axes[0,0].set_xlabel('Return (%)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Trade Return Distribution')
axes[0,0].legend(fontsize=9)
axes[0,0].grid(True, alpha=0.3)

# 2. Cumulative P&L
cum_pnl = trades['PnL'].cumsum()
axes[0,1].plot(range(len(cum_pnl)), cum_pnl, color=STRAT_COLOR, linewidth=1.5)
axes[0,1].fill_between(range(len(cum_pnl)), cum_pnl, 0, 
                        where=cum_pnl >= 0, color='#2ECC71', alpha=0.2)
axes[0,1].fill_between(range(len(cum_pnl)), cum_pnl, 0, 
                        where=cum_pnl < 0, color='#E74C3C', alpha=0.2)
axes[0,1].set_xlabel('Trade #')
axes[0,1].set_ylabel('Cumulative P&L ($)')
axes[0,1].set_title('Cumulative P&L by Trade')
axes[0,1].grid(True, alpha=0.3)

# 3. Monthly returns heatmap
port_rets = pf.returns()
monthly = port_rets.resample('ME').apply(lambda x: (1+x).prod() - 1)
monthly_pivot = pd.DataFrame({
    'Year': monthly.index.year,
    'Month': monthly.index.month,
    'Return': monthly.values
}).pivot_table(values='Return', index='Year', columns='Month', aggfunc='first')
monthly_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

im = axes[1,0].imshow(monthly_pivot.values * 100, cmap='RdYlGn', aspect='auto', vmin=-10, vmax=10)
axes[1,0].set_xticks(range(len(monthly_pivot.columns)))
axes[1,0].set_xticklabels(monthly_pivot.columns, fontsize=7)
axes[1,0].set_yticks(range(len(monthly_pivot.index)))
axes[1,0].set_yticklabels(monthly_pivot.index, fontsize=8)
for yi in range(len(monthly_pivot.index)):
    for xi in range(len(monthly_pivot.columns)):
        val = monthly_pivot.values[yi, xi]
        if not np.isnan(val):
            axes[1,0].text(xi, yi, f'{val*100:.1f}', ha='center', va='center', fontsize=7,
                          color='white' if abs(val) > 0.05 else 'black')
axes[1,0].set_title('Monthly Returns (%)')

# 4. Holding period distribution
if 'Duration' in trades.columns:
    durations = pd.to_timedelta(trades['Duration']).dt.days
    axes[1,1].hist(durations, bins=20, color=STRAT_COLOR, alpha=0.8)
    axes[1,1].set_xlabel('Holding Period (days)')
    axes[1,1].set_ylabel('Count')
    axes[1,1].set_title(f'Holding Period (avg: {durations.mean():.0f}d)')
    axes[1,1].grid(True, alpha=0.3)
else:
    axes[1,1].text(0.5, 0.5, 'Duration data\nnot available', ha='center', va='center', fontsize=12)
    axes[1,1].set_title('Holding Period')

plt.suptitle(f'{TICKER} \u2014 Supertrend Trade Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Monte Carlo FTMO Simulation (10,000 paths)
# ============================================================
N_SIMS = 10_000
trade_returns = trades['Return'].values

# Simulate equity paths by resampling trades
np.random.seed(42)
sim_equity = np.ones((N_SIMS, len(trade_returns) + 1)) * INIT_CASH

for i in range(N_SIMS):
    shuffled = np.random.choice(trade_returns, size=len(trade_returns), replace=True)
    sim_equity[i, 1:] = INIT_CASH * np.cumprod(1 + shuffled)

# FTMO rules
FTMO_TARGET = INIT_CASH * 1.10  # +10%
FTMO_MAX_LOSS = INIT_CASH * 0.90  # -10% total
hit_target = (sim_equity.max(axis=1) >= FTMO_TARGET).mean() * 100
hit_loss = (sim_equity.min(axis=1) <= FTMO_MAX_LOSS).mean() * 100
pass_rate = ((sim_equity.max(axis=1) >= FTMO_TARGET) & (sim_equity.min(axis=1) > FTMO_MAX_LOSS)).mean() * 100

# Verdict
if pass_rate >= 60:
    verdict = "\u2705 PASS"
elif pass_rate >= 30:
    verdict = "\u26a0\ufe0f CONDITIONAL"
else:
    verdict = "\u274c FAIL"

fig, ax = plt.subplots(figsize=(14, 5))

# Plot percentile bands
for pct in [5, 25, 50, 75, 95]:
    line = np.percentile(sim_equity, pct, axis=0)
    alpha = 0.3 if pct in [5, 95] else 0.5 if pct in [25, 75] else 1.0
    lw = 0.8 if pct != 50 else 2.0
    style = '--' if pct != 50 else '-'
    ax.plot(range(len(line)), line, linewidth=lw, linestyle=style, alpha=alpha,
            color=STRAT_COLOR, label=f'P{pct}')

ax.axhline(FTMO_TARGET, color='#2ECC71', linestyle='--', linewidth=1, label='FTMO Target (+10%)')
ax.axhline(FTMO_MAX_LOSS, color='#E74C3C', linestyle='--', linewidth=1, label='FTMO Max Loss (-10%)')
ax.fill_between(range(sim_equity.shape[1]),
                np.percentile(sim_equity, 5, axis=0),
                np.percentile(sim_equity, 95, axis=0),
                color=STRAT_COLOR, alpha=0.1)

ax.set_xlabel('Trade #', fontsize=11)
ax.set_ylabel('Equity ($)', fontsize=11)
ax.set_title(
    f'{TICKER} \u2014 Monte Carlo FTMO Simulation ({N_SIMS:,} paths)\n'
    f'Pass Rate: {pass_rate:.1f}%  |  Verdict: {verdict}',
    fontsize=13, fontweight='bold'
)
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"FTMO Monte Carlo Results:")
print(f"  Hit +10% target: {hit_target:.1f}%")
print(f"  Hit -10% limit:  {hit_loss:.1f}%")
print(f"  Clean pass rate: {pass_rate:.1f}%")
print(f"  Verdict: {verdict}")

In [ ]:
# ============================================================
# Log to Google Sheets & Export
# ============================================================
try:
    sys.path.insert(0, os.path.join(os.getcwd(), 'lib'))
    from sheets_logger import SheetsLogger
    
    logger = SheetsLogger()
    logger.log_result(
        ticker=TICKER,
        strategy='Supertrend',
        strategy_family='Trend_Following',
        fast_param=BEST_ATR,
        slow_param=BEST_MULT,
        filter_param=None,
        is_sharpe=float(best['is_sharpe']),
        is_return=None,
        is_max_dd=None,
        is_trades=None,
        oos_sharpe=float(best['oos_sharpe']),
        oos_return=float(best['oos_return']) if 'oos_return' in best else None,
        oos_max_dd=float(best['oos_max_dd']) if 'oos_max_dd' in best else None,
        oos_trades=int(best['oos_trades']) if 'oos_trades' in best else None,
        full_sharpe=sharpe,
        full_return=total_ret,
        full_max_dd=max_dd,
        full_trades=n_trades,
        win_rate=win_rate,
        profit_factor=pf_ratio,
        expectancy=None,
        payoff_ratio=None,
        sharpe_degradation=float(best['degradation']) if 'degradation' in best else None,
        bh_sharpe=bh_sharpe,
        bh_return=bh_ret,
        mc_ftmo_pass_rate=pass_rate,
        mc_verdict=verdict,
        data_start=START_DATE,
        data_end=close.index[-1].strftime('%Y-%m-%d'),
        total_bars=len(close),
        notes=f"ATR={BEST_ATR} Mult={BEST_MULT}"
    )
    print("\u2705 Results logged to Google Sheets")
except Exception as e:
    print(f"\u26a0\ufe0f Sheets logging skipped: {e}")

# Export
try:
    STRATEGY_NAME = "Supertrend"
    PARAM_COLS = ["atr_period", "multiplier"]
    exec(open(os.path.join('lib', 'UNIVERSAL_EXPORT_CELL_v2.py'), encoding='utf-8').read())
except Exception as e:
    print(f"\u26a0\ufe0f Export skipped: {e}")